In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class DataPreprocessor:
    def __init__(self):
        self.encoders = {}
        self.scaler = StandardScaler()
        self.num_cols = []
        self.cat_cols = []
        self.target_col = None
        self.trained_columns = None

    def detect_column_types(self, df, target_col):
        """Auto-detect numerical and categorical columns"""
        self.target_col = target_col
        for col in df.columns:
            if col == target_col:
                continue
            if df[col].dtype in ['int64', 'float64'] and df[col].nunique() > 20:
                self.num_cols.append(col)
            else:
                self.cat_cols.append(col)
        print(f"✓ Detected {len(self.num_cols)} numerical, {len(self.cat_cols)} categorical features")

    def handle_missing(self, df):
        """Intelligent missing value imputation"""
        for col in self.num_cols:
            if df[col].isnull().any():
                df[col].fillna(df[col].median(), inplace=True)
        for col in self.cat_cols:
            if df[col].isnull().any():
                df[col].fillna(df[col].mode()[0], inplace=True)
        return df

    def encode_features(self, df, fit=True):
        """Target encoding for high cardinality, one-hot for low"""
        for col in self.cat_cols:
            cardinality = df[col].nunique()
            if cardinality <= 10:  # One-hot encoding
                dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
                df = pd.concat([df, dummies], axis=1)
                df.drop(col, axis=1, inplace=True)
            else:  # Target encoding
                if fit:
                    target_mean = df.groupby(col)[self.target_col].mean()
                    self.encoders[col] = target_mean
                df[col] = df[col].map(self.encoders[col]).fillna(self.encoders[col].mean())
        return df

    def fit_transform(self, df, target_col):
        """Complete preprocessing pipeline"""
        self.detect_column_types(df, target_col)
        df = self.handle_missing(df)
        df = self.encode_features(df, fit=True)

        X = df.drop(target_col, axis=1)
        y = df[target_col]

        # Store columns after fitting
        self.trained_columns = X.columns

        # Scale numerical features
        X[self.num_cols] = self.scaler.fit_transform(X[self.num_cols])
        print(f"✓ Preprocessing complete. Final shape: {X.shape}")
        return X, y

    def transform(self, df):
        """Transform new data using fitted preprocessing"""
        df = self.handle_missing(df)
        df = self.encode_features(df, fit=False)
        X = df.drop(self.target_col, axis=1, errors='ignore')

        # Ensure columns match the training data after one-hot encoding
        X = X.reindex(columns=self.trained_columns, fill_value=0)

        # Scale numerical features
        X[self.num_cols] = self.scaler.transform(X[self.num_cols])
        return X

class ModelZoo:
    @staticmethod
    def get_models():
        return {
            'RandomForest': {
                'model': RandomForestClassifier,
                'space': {
                    'n_estimators': (50, 300),
                    'max_depth': (3, 15),
                    'min_samples_split': (2, 20),
                    'min_samples_leaf': (1, 10)
                }
            },
            'XGBoost': {
                'model': xgb.XGBClassifier,
                'space': {
                    'n_estimators': (50, 300),
                    'max_depth': (3, 10),
                    'learning_rate': (0.01, 0.3),
                    'subsample': (0.6, 1.0),
                    'colsample_bytree': (0.6, 1.0)
                }
            },
            'LightGBM': {
                'model': lgb.LGBMClassifier,
                'space': {
                    'n_estimators': (50, 300),
                    'max_depth': (3, 10),
                    'learning_rate': (0.01, 0.3),
                    'num_leaves': (20, 100)
                }
            },
            'CatBoost': {
                'model': CatBoostClassifier,
                'space': {
                    'iterations': (50, 300),
                    'depth': (3, 10),
                    'learning_rate': (0.01, 0.3),
                    'l2_leaf_reg': (1, 10)
                }
            }
        }

class HyperparameterTuner:
    def __init__(self, X, y, n_trials=20, cv_folds=5):
        self.X = X
        self.y = y
        self.n_trials = n_trials
        self.cv_folds = cv_folds
        self.results = []

    def objective(self, trial, model_name, model_class, param_space):

        params = {}
        for param, bounds in param_space.items():
            if isinstance(bounds[0], int):
                params[param] = trial.suggest_int(param, bounds[0], bounds[1])
            else:
                params[param] = trial.suggest_float(param, bounds[0], bounds[1])

        skf = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
        scores = []

        for train_idx, val_idx in skf.split(self.X, self.y):
            X_train, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_train, y_val = self.y.iloc[train_idx], self.y.iloc[val_idx]

            if model_name == 'CatBoost':
                model = model_class(**params, verbose=0, random_state=42)
            else:
                model = model_class(**params, random_state=42, verbose=0)

            model.fit(X_train, y_train)

            y_pred = model.predict_proba(X_val)[:, 1]
            scores.append(roc_auc_score(y_val, y_pred))

        mean_score = np.mean(scores)
        return mean_score

    def tune_model(self, model_name, model_class, param_space):
        print(f"\n🔍 Tuning {model_name}...")

        study = optuna.create_study(direction='maximize', study_name=model_name)
        study.optimize(
            lambda trial: self.objective(trial, model_name, model_class, param_space),
            n_trials=self.n_trials,
            show_progress_bar=False
        )

        best_params = study.best_params
        best_score = study.best_value

        print(f"✓ Best AUC: {best_score:.4f}")

        self.results.append({
            'model': model_name,
            'params': best_params,
            'score': best_score
        })

        return best_params, best_score

    def find_best_model(self):
        models = ModelZoo.get_models()

        for model_name, config in models.items():
            self.tune_model(model_name, config['model'], config['space'])

        best_result = max(self.results, key=lambda x: x['score'])
        print(f"\n🏆 Best Model: {best_result['model']} (AUC: {best_result['score']:.4f})")

        return best_result

class ExplainabilityEngine:
    def __init__(self, model, X, feature_names):
        self.model = model
        self.X = X
        self.feature_names = feature_names

    def generate_shap_explanations(self):
        print("\n📊 Generating SHAP explanations...")

        explainer = shap.TreeExplainer(self.model)
        shap_values = explainer.shap_values(self.X)

        if isinstance(shap_values, list):
            shap_values = shap_values[1] if len(shap_values) == 2 else shap_values[0]

        if len(shap_values.shape) == 3:
            shap_values = shap_values[:, :, 1]

        if len(shap_values.shape) != 2:
            shap_values = shap_values.reshape(len(self.X), -1)

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, self.X, feature_names=self.feature_names, show=False)
        plt.title("SHAP Feature Importance")
        plt.tight_layout()
        plt.savefig('shap_global_importance.png', dpi=300, bbox_inches='tight')
        plt.close()

        sample_indices = np.random.choice(len(self.X), min(5, len(self.X)), replace=False)

        if isinstance(explainer.expected_value, (list, np.ndarray)):
            base_value = float(explainer.expected_value[1] if len(explainer.expected_value) > 1 else explainer.expected_value[0])
        else:
            base_value = float(explainer.expected_value)

        for i, idx in enumerate(sample_indices):
            try:
                plt.figure(figsize=(10, 6))

                explanation = shap.Explanation(
                    values=shap_values[idx],
                    base_values=base_value,
                    data=self.X.iloc[idx].values,
                    feature_names=self.feature_names
                )

                shap.waterfall_plot(explanation, show=False)
                plt.title(f"SHAP Local Explanation - Sample {i+1}")
                plt.tight_layout()
                plt.savefig(f'shap_local_sample_{i+1}.png', dpi=300, bbox_inches='tight')
                plt.close()
            except Exception as e:
                print(f"Warning: Could not generate waterfall plot for sample {i+1}: {str(e)}")
                plt.close()
                continue

        print("✓ SHAP plots saved")

        importance = pd.DataFrame({
            'feature': self.feature_names,
            'importance': np.abs(shap_values).mean(axis=0)
        }).sort_values('importance', ascending=False)

        return importance

class DriftMonitor:
    @staticmethod
    def calculate_psi(expected, actual, bins=10):
        expected_counts, bin_edges = np.histogram(expected, bins=bins)
        actual_counts, _ = np.histogram(actual, bins=bin_edges)

        expected_pct = expected_counts / len(expected)
        actual_pct = actual_counts / len(actual)

        expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
        actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

        psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
        return psi

    @staticmethod
    def calculate_kl_divergence(expected, actual, bins=10):
        expected_counts, bin_edges = np.histogram(expected, bins=bins)
        actual_counts, _ = np.histogram(actual, bins=bin_edges)

        expected_pct = expected_counts / len(expected) + 1e-10
        actual_pct = actual_counts / len(actual) + 1e-10

        kl_div = np.sum(actual_pct * np.log(actual_pct / expected_pct))
        return kl_div

    def monitor_drift(self, X_train, X_test):
        """Monitor drift across all features"""
        print("\n🔍 Monitoring data drift...")

        drift_results = []
        for col in X_train.columns:
            psi = self.calculate_psi(X_train[col].values, X_test[col].values)
            kl_div = self.calculate_kl_divergence(X_train[col].values, X_test[col].values)

            drift_results.append({
                'feature': col,
                'PSI': psi,
                'KL_Divergence': kl_div,
                'drift_status': 'High' if psi > 0.2 else 'Medium' if psi > 0.1 else 'Low'
            })

        drift_df = pd.DataFrame(drift_results).sort_values('PSI', ascending=False)
        print("\nTop 5 Features with Highest Drift:")
        print(drift_df.head())

        return drift_df

class ReportGenerator:
    @staticmethod
    def generate_html_report(best_model_info, feature_importance, drift_results, training_time):
        html = f"""
        <html>
        <head>
            <title>AutoML Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
                h1 {{ color: #2c3e50; }}
                h2 {{ color: #34495e; margin-top: 30px; }}
                table {{ border-collapse: collapse; width: 100%; margin: 20px 0; background: white; }}
                th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
                th {{ background-color: #3498db; color: white; }}
                .metric {{ background: white; padding: 20px; margin: 10px 0; border-radius: 5px; }}
                .highlight {{ color: #e74c3c; font-weight: bold; }}
            </style>
        </head>
        <body>
            <h1>🤖 AutoML System Report</h1>
            <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

            <h2>🏆 Best Model</h2>
            <div class="metric">
                <p><strong>Model:</strong> {best_model_info['model']}</p>
                <p><strong>AUC Score:</strong> {best_model_info['score']:.4f}</p>
                <p><strong>Training Time:</strong> {training_time:.2f} seconds</p>
            </div>

            <h2>⚙️ Best Hyperparameters</h2>
            <table>
                <tr><th>Parameter</th><th>Value</th></tr>
                {''.join(f"<tr><td>{k}</td><td>{v}</td></tr>" for k, v in best_model_info['params'].items())}
            </table>

            <h2>📊 Top 10 Feature Importances</h2>
            <table>
                <tr><th>Rank</th><th>Feature</th><th>Importance</th></tr>
                {''.join(f"<tr><td>{i+1}</td><td>{row['feature']}</td><td>{row['importance']:.4f}</td></tr>"
                         for i, row in feature_importance.head(10).iterrows())}
            </table>

            <h2>🔍 Drift Analysis</h2>
            <table>
                <tr><th>Feature</th><th>PSI</th><th>KL Divergence</th><th>Status</th></tr>
                {''.join(f"<tr><td>{row['feature']}</td><td>{row['PSI']:.4f}</td><td>{row['KL_Divergence']:.4f}</td><td class='{'highlight' if row['drift_status'] == 'High' else ''}'>{row['drift_status']}</td></tr>"
                         for _, row in drift_results.head(10).iterrows())}
            </table>

            <h2>📈 SHAP Visualizations</h2>
            <p>SHAP plots have been saved to:</p>
            <ul>
                <li>shap_global_importance.png</li>
                <li>shap_local_sample_1.png to shap_local_sample_5.png</li>
            </ul>
        </body>
        </html>
        """

        with open('automl_report.html', 'w') as f:
            f.write(html)

        print("\n✓ Report saved as 'automl_report.html'")

class AutoML:
    def __init__(self, n_trials=20, cv_folds=5):
        self.n_trials = n_trials
        self.cv_folds = cv_folds
        self.preprocessor = DataPreprocessor()
        self.best_model = None
        self.best_model_info = None

    def fit(self, df, target_col, test_size=0.2):
        start_time = datetime.now()
        print("=" * 60)
        print(" AutoML Pipeline Started")
        print("=" * 60)

        print("\n STAGE 1: Data Preprocessing")
        X, y = self.preprocessor.fit_transform(df, target_col)

        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

        print("\n STAGE 2-3: Model Selection & Hyperparameter Tuning")
        tuner = HyperparameterTuner(X_train, y_train, self.n_trials, self.cv_folds)
        self.best_model_info = tuner.find_best_model()

        model_class = ModelZoo.get_models()[self.best_model_info['model']]['model']
        # Directly assign the fitted model to self.best_model
        self.best_model = model_class(**self.best_model_info['params'], verbose=0, random_state=42)
        self.best_model.fit(X_train, y_train)


        print("\n STAGE 4: Model Explainability")
        explainer = ExplainabilityEngine(self.best_model, X_train, X.columns.tolist())
        feature_importance = explainer.generate_shap_explanations()

        print("\n STAGE 5: Drift Monitoring")
        drift_monitor = DriftMonitor()
        drift_results = drift_monitor.monitor_drift(X_train, X_test)

        print("\n STAGE 6: Report Generation")
        training_time = (datetime.now() - start_time).total_seconds()
        ReportGenerator.generate_html_report(
            self.best_model_info,
            feature_importance,
            drift_results,
            training_time
        )

        print("\n" + "=" * 60)
        print(" AutoML Pipeline Completed!")
        print(f"  Total Time: {training_time:.2f} seconds")
        print("=" * 60)

        return self

    def predict(self, data):

        if isinstance(data, str):
            print(f" Loading data from {data}")
            df = pd.read_csv(data)
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
        elif isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, pd.DataFrame):
            df = data.copy()
        else:
            raise ValueError("Input must be DataFrame, CSV path, dict, or list of dicts")

        # Preprocess and predict
        X_processed = self.preprocessor.transform(df)
        predictions = self.best_model.predict(X_processed)

        print(f"✓ Generated {len(predictions)} predictions")
        return predictions

    def predict_proba(self, data):

        if isinstance(data, str):
            print(f" Loading data from {data}")
            df = pd.read_csv(data)
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
        elif isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, pd.DataFrame):
            df = data.copy()
        else:
            raise ValueError("Input must be DataFrame, CSV path, dict, or list of dicts")

        X_processed = self.preprocessor.transform(df)
        probabilities = self.best_model.predict_proba(X_processed)

        print(f"✓ Generated {len(probabilities)} probability predictions")
        return probabilities

    def predict_with_explanation(self, data, output_file='predictions.csv'):

        predictions = self.predict(data)
        probabilities = self.predict_proba(data)

        results = pd.DataFrame({
            'prediction': predictions,
            'probability_class_0': probabilities[:, 0],
            'probability_class_1': probabilities[:, 1],
            'confidence': probabilities.max(axis=1)
        })

        results.to_csv(output_file, index=False)
        print(f"Predictions saved to {output_file}")

        return results

if __name__ == "__main__":

     df = pd.read_csv('/content/heart_attack_prediction_dataset.csv')

     automl = AutoML(n_trials=10, cv_folds=3)
     automl.fit(df, target_col='Heart Attack Risk', test_size=0.2)

     print("\n2️  Predicting from CSV file:")
     test_df = df.drop('Heart Attack Risk', axis=1).sample(10, random_state=42)
     test_df.to_csv('test_data.csv', index=False)
     predictions = automl.predict('test_data.csv')
     print(f"Predictions: {predictions}")

 AutoML Pipeline Started

 STAGE 1: Data Preprocessing
✓ Detected 8 numerical, 17 categorical features


[I 2025-10-28 04:36:03,335] A new study created in memory with name: RandomForest


✓ Preprocessing complete. Final shape: (8763, 49)

 STAGE 2-3: Model Selection & Hyperparameter Tuning

🔍 Tuning RandomForest...


[I 2025-10-28 04:36:16,628] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 246, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 2}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:36:26,523] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 261, 'max_depth': 12, 'min_samples_split': 16, 'min_samples_leaf': 9}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:36:33,580] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 234, 'max_depth': 11, 'min_samples_split': 19, 'min_samples_leaf': 6}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:36:42,096] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 271, 'max_depth': 8, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:36:48,820] Trial 4 finished with value: 1.0 and parameters: {'n_estimators': 222, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 7}. Best is trial 0 with value: 1.0.
[I 2025-10-28 

✓ Best AUC: 1.0000

🔍 Tuning XGBoost...


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped t

[I 2025-10-28 04:37:26,542] Trial 9 finished with value: 1.0 and parameters: {'n_estimators': 96, 'max_depth': 7, 'learning_rate': 0.05391512042987581, 'num_leaves': 97}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:37:26,545] A new study created in memory with name: CatBoost


✓ Best AUC: 1.0000

🔍 Tuning CatBoost...


[I 2025-10-28 04:37:27,972] Trial 0 finished with value: 1.0 and parameters: {'iterations': 147, 'depth': 5, 'learning_rate': 0.2730488539937085, 'l2_leaf_reg': 2}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:37:33,283] Trial 1 finished with value: 1.0 and parameters: {'iterations': 227, 'depth': 7, 'learning_rate': 0.1965167808707177, 'l2_leaf_reg': 8}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:37:35,419] Trial 2 finished with value: 1.0 and parameters: {'iterations': 242, 'depth': 5, 'learning_rate': 0.241730571013025, 'l2_leaf_reg': 7}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:37:36,619] Trial 3 finished with value: 1.0 and parameters: {'iterations': 159, 'depth': 4, 'learning_rate': 0.1282758489409415, 'l2_leaf_reg': 3}. Best is trial 0 with value: 1.0.
[I 2025-10-28 04:37:40,544] Trial 4 finished with value: 1.0 and parameters: {'iterations': 231, 'depth': 7, 'learning_rate': 0.13816405485428757, 'l2_leaf_reg': 2}. Best is trial 0 with value: 1.0.
[I 2025-10

✓ Best AUC: 1.0000

🏆 Best Model: RandomForest (AUC: 1.0000)

 STAGE 4: Model Explainability

📊 Generating SHAP explanations...
✓ SHAP plots saved

 STAGE 5: Drift Monitoring

🔍 Monitoring data drift...

Top 5 Features with Highest Drift:
                   feature       PSI  KL_Divergence drift_status
3           Blood Pressure  0.018036       0.008783          Low
7                   Income  0.008255       0.004076          Low
1                      Age  0.006859       0.003424          Low
9            Triglycerides  0.006563       0.003274          Low
6  Sedentary Hours Per Day  0.005871       0.002927          Low

 STAGE 6: Report Generation

✓ Report saved as 'automl_report.html'

 AutoML Pipeline Completed!
  Total Time: 135.10 seconds

2️  Predicting from CSV file:
 Loading data from test_data.csv
✓ Generated 10 predictions
Predictions: [0 0 0 0 0 0 0 0 0 0]


In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 5.0 MB/s eta 0:00:00


In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.6 MB/s eta 0:00:00
